# 섹션3-6. AI를 활용한 시계열 데이터(주식)의 Regression, Waterfall Chart, Line Chart, Candle Chart, 이동평균선

> 강의: [32가지 데이터 시각화 전략 - 비전공자를 위한 기초이론 & 실습](https://www.inflearn.com/course/32-data-visualizatio/dashboard?cid=343563) (반병현) — 전체 17강

- [x] 강의 시청 완료
- [x] 실습/정리 완료

## 배운 내용

<!-- 강의를 보면서 핵심을 적는다 -->

-

## 목표 / 재현할 것

<!-- 이 강의에서 만든 차트를 내 방식대로 다시 만들어본다 -->

-


## 실습

> **데이터:** 강의 제공 애플 주가 CSV(2010~2020, 2,517행). `.gitignore`로 커밋에서
> 제외했다 — clone한 경우 이 노트북의 데이터 로딩 셀은 파일이 없어 실패한다.
> 강의와 무관하게 재현하려면 같은 기간의 공개 주가 데이터를 받아 컬럼명만 맞추면 된다.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_utils import setup, load_sample

setup()
rng = np.random.default_rng(0)

df = pd.read_csv("../references/lecture_data/애플 주가/애플 주가.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)
print(f"{len(df):,}행, {df.Date.min().date()} ~ {df.Date.max().date()}")
df.head(3)

### 1. 산점도 + 회귀선 — 엑셀의 '추세선'

In [ ]:
# 엑셀 산점도는 날짜를 1899-12-30 기준 일련번호로 취급한다
x = (df["Date"] - pd.Timestamp("1899-12-30")).dt.days.values.astype(float)
y = df["Close"].values

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(x[::5], y[::5], s=4, alpha=0.3, color="#4C78A8", label="종가(5일 간격 표시)")

coef = np.polyfit(x, np.log(y), 1)  # 지수 추세선: ln(y)에 선형회귀
gx = np.linspace(x.min(), x.max(), 200)
ax.plot(gx, np.exp(np.polyval(coef, gx)), color="#E45756", lw=2, label="지수 추세선")
ax.set(xlabel="날짜(일련번호)", ylabel="종가($)", title="애플 주가 10년 + 지수 추세선")
ax.set_xticks([x.min(), x.max()], [df.Date.min().year, df.Date.max().year])
ax.legend()
plt.show()

### 2. R² 비교 — 같은 기준으로 재야 한다

In [ ]:
def r2(y_true, y_pred):
    ss_res = ((y_true - y_pred) ** 2).sum()
    ss_tot = ((y_true - y_true.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot


results = {}
results["선형"] = r2(y, np.polyval(np.polyfit(x, y, 1), x))
results["로그"] = r2(y, np.polyval(np.polyfit(np.log(x), y, 1), np.log(x)))
results["다항식(2차)"] = r2(y, np.polyval(np.polyfit(x, y, 2), x))

# 엑셀은 지수·거듭제곱의 R²를 로그 변환된 공간에서 계산해 표시한다 — 원래 스케일 R²와 다르다
exp_coef = np.polyfit(x, np.log(y), 1)
results["지수 (엑셀 표시값, 로그공간)"] = r2(np.log(y), np.polyval(exp_coef, x))
results["지수 (원래 스케일)"] = r2(y, np.exp(np.polyval(exp_coef, x)))

pow_coef = np.polyfit(np.log(x), np.log(y), 1)
results["거듭제곱 (엑셀 표시값, 로그공간)"] = r2(np.log(y), np.polyval(pow_coef, np.log(x)))
results["거듭제곱 (원래 스케일)"] = r2(y, np.exp(np.polyval(pow_coef, np.log(x))))

for k, v in results.items():
    print(f"{k:32} R² = {v:.4f}")

> **함정.** 엑셀에서 지수·거듭제곱 추세선의 R²는 `ln(y)`를 기준으로 계산되는데,
> 선형·다항식 R²는 원래 스케일 기준이다. **단위가 다른 값을 나란히 놓고 "지수가 가장
> 높다"고 비교하면 안 된다.** 원래 스케일로 통일하면 순위가 바뀐다 — 아래 셀에서 확인.

In [ ]:
fair = {
    "다항식(2차)": results["다항식(2차)"],
    "지수": results["지수 (원래 스케일)"],
    "거듭제곱": results["거듭제곱 (원래 스케일)"],
    "선형": results["선형"],
    "로그": results["로그"],
}
ranked = sorted(fair.items(), key=lambda kv: -kv[1])
for name, v in ranked:
    print(f"{name:10} {v:.4f}")
print()
print("같은 기준으로 재면 다항식이 근소하게 앞선다 - 강의에서 \"지수가 가장 잘 맞는다\"고 한 것과 다르다.")
print("다만 0.898 vs 0.896 차이로 모델을 고르는 것 자체가 사실 의미 없다.")

### 3. R²가 높다고 예측이 맞는 건 아니다 — 홀드아웃 검증

In [ ]:
train = df["Date"] < "2019-01-01"
co = np.polyfit(x[train.values], np.log(y[train.values]), 1)

train_r2 = r2(np.log(y[train.values]), np.polyval(co, x[train.values]))
pred = np.exp(np.polyval(co, x[~train.values]))
actual = y[~train.values]
mape = np.mean(np.abs(pred - actual) / actual) * 100

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, y, color="#4C78A8", lw=1.2, label="실제 종가")
gx2 = np.linspace(x.min(), x.max(), 200)
ax.plot(gx2, np.exp(np.polyval(co, gx2)), color="#E45756", ls="--", lw=1.6, label="2018년까지 학습한 추세선")
split = x[train.values][-1]
ax.axvline(split, color="gray", ls=":", lw=1)
ax.text(split, y.max() * 0.95, " ← 학습 | 예측 →", fontsize=9, color="gray")
ax.set(xlabel="날짜", ylabel="종가($)", title="2018년까지 학습 → 2019~2020 예측")
ax.set_xticks([x.min(), x.max()], [df.Date.min().year, df.Date.max().year])
ax.legend()
plt.show()

print(f"학습 구간 R²(로그공간): {train_r2:.4f}")
print(f"예측 구간 평균절대오차율(MAPE): {mape:.1f}%")
print(f"2020-07-10 실제 ${actual[-1]:.1f}  vs  예측 ${pred[-1]:.1f}  "
      f"(실제가 예측의 {actual[-1]/pred[-1]:.2f}배)")

> 학습 R²는 0.88로 높았지만 실제 2020년 종가는 예측의 **1.5배**였다. 주가처럼
> 우상향하는 시계열은 어떤 매끄러운 증가 곡선을 갖다 대도 R²가 높게 나온다 — R²는
> "데이터를 얼마나 설명하는가"이지 "미래를 얼마나 맞히는가"가 아니다.

### 4. 워터폴 차트 — 연간 증감

In [ ]:
year_end = df.set_index("Date")["Close"].resample("YE").last()
year_end.index = year_end.index.year
diffs = year_end.diff().dropna()

fig, ax = plt.subplots(figsize=(9, 4.5))
running = year_end.iloc[0]
ax.bar(0, running, color="lightgray", label="시작(2010년 종가)")
for i, (yr, d) in enumerate(diffs.items(), start=1):
    bottom = running if d < 0 else running
    color = "#4C78A8" if d > 0 else "#E45756"
    ax.bar(i, d, bottom=min(running, running + d), color=color)
    running += d
ax.set_xticks(range(len(year_end)), year_end.index)
ax.set(ylabel="종가($)", title="연간 종가 증감 (워터폴)")
plt.show()

print("연간 증감:")
for yr, d in diffs.items():
    print(f"  {yr}: {d:+.1f}" + ("  ← 감소" if d < 0 else ""))

> 강의에서는 "증가하기만 하는 워터폴"이라고 했지만 실제로는 **2015년(-5.1), 2018년(-11.5)
> 두 해가 감소**했다. 워터폴은 누적 방향을 보여줄 뿐 방향이 항상 한쪽인지는 데이터에 달렸다.

### 5. 캔들스틱 + 이동평균선

In [ ]:
year_open = df.set_index("Date")["Open"].resample("YE").first(); year_open.index = year_open.index.year
year_low = df.set_index("Date")["Low"].resample("YE").min(); year_low.index = year_low.index.year
year_high = df.set_index("Date")["High"].resample("YE").max(); year_high.index = year_high.index.year

fig, ax = plt.subplots(figsize=(9, 4.5))
for i, yr in enumerate(year_end.index):
    o, c, lo, hi = year_open[yr], year_end[yr], year_low[yr], year_high[yr]
    color = "#4C78A8" if c >= o else "#E45756"
    ax.plot([i, i], [lo, hi], color=color, lw=1)
    ax.bar(i, abs(c - o), bottom=min(o, c), width=0.5, color=color)
ax.set_xticks(range(len(year_end)), year_end.index)
ax.set(ylabel="종가($)", title="연단위 캔들스틱 (OHLC)")
plt.show()

# 이동평균선 — 2017년 이후 일간 데이터로
recent = df[df["Date"] >= "2017-01-01"]
ma120 = recent["Close"].rolling(120).mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(recent["Date"], recent["Close"], lw=0.7, alpha=0.6, label="일간 종가")
ax.plot(recent["Date"], ma120, lw=2, color="#E45756", label="120일 이동평균")
ax.set(ylabel="종가($)", title="일간 종가 + 120일 이동평균 (2017~2020)")
ax.legend()
plt.show()

---

## 메모

-
